<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/03_extract_2wiki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip -q install groq

In [16]:
import json, re, time
from pathlib import Path
from google.colab import drive, userdata
from groq import Groq, RateLimitError, BadRequestError

In [17]:
drive.mount("/content/drive")
WORKDIR = Path("/content/drive/MyDrive/HybridGraphRAG/twowiki")

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

chunks = load_jsonl(WORKDIR / "document_chunks.jsonl")
questions = load_jsonl(WORKDIR / "questions.jsonl")

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
LLM = "openai/gpt-oss-20b"   # the real extraction model, not the safeguard one

def chat_with_retry(**kwargs):
    for attempt in range(8):
        try:
            return groq_client.chat.completions.create(**kwargs)
        except RateLimitError:
            wait = 3 * (attempt + 1)
            print(f"  rate limited, waiting {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after 8 retries")

print(len(chunks), "chunks,", len(questions), "questions")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
182 chunks, 20 questions


In [18]:
ENTITY_TYPES = {
    "PERSON":       "a real person",
    "WORK":         "a film, song, book, album or other creative work",
    "PLACE":        "a city, region, country or other location",
    "ORGANIZATION": "a company, school, band, club or institution",
    "AWARD":        "a prize or honour",
    "DATE":         "a date or year, copied as written in the text",
    "OTHER":        "important, but fits none of the above",
}

# Direction rule: (source, RELATION, target). The definitions say who is who.
RELATION_TYPES = {
    # links between things (these are what connect articles to each other)
    "DIRECTOR":               "target directed source (source is a film)",
    "FATHER":                 "target is the father of source",
    "MOTHER":                 "target is the mother of source",
    "SPOUSE":                 "target is married to source",
    "CHILD":                  "target is a child of source",
    "SIBLING":                "target is a brother or sister of source",
    "PERFORMER":              "target performed source (e.g. the singer of a song)",
    "COMPOSER":               "target composed source",
    "EDUCATED_AT":            "source studied at target",
    "EMPLOYER":               "source works or worked for target",
    "AWARD_RECEIVED":         "source received target (an award)",
    "PLACE_OF_BIRTH":         "source was born in target",
    "PLACE_OF_DEATH":         "source died in target",
    "PLACE_OF_BURIAL":        "source was buried in target",
    "COUNTRY_OF_CITIZENSHIP": "source is a citizen of target",
    "COUNTRY_OF_ORIGIN":      "source (a work) comes from target (a country)",
    "COUNTRY":                "source is located in target (a country)",
    # facts whose target is a DATE
    "DATE_OF_BIRTH":          "source was born on target (a DATE)",
    "DATE_OF_DEATH":          "source died on target (a DATE)",
    "PUBLICATION_DATE":       "source was released or published on target (a DATE)",
    "INCEPTION":              "source was founded or created on target (a DATE)",
    # catch-all
    "RELATED_TO":             "a clear link that fits none of the above",
}
print(len(ENTITY_TYPES), "entity types,", len(RELATION_TYPES), "relation types")

7 entity types, 22 relation types


In [19]:
def extract_with_retry(text, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        try:
            result = extract(text)
            return normalize_extraction(result)

        except BadRequestError as e:
            print(
                f"Structured-output failure "
                f"(attempt {attempt}/{max_attempts}): {e}"
            )

            if attempt < max_attempts:
                time.sleep(2 * attempt)

        except Exception as e:
            print(f"Unexpected extraction error: {e}")
            raise

    raise RuntimeError(
        "Extraction failed after all retry attempts."
    )

In [42]:
import json, re

def build_schema(entity_types, relation_types):
    return {
        "type": "object",
        "properties": {
            "entities": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string"},
                        "type": {"type": "string", "enum": list(entity_types)},
                    },
                    "required": ["name", "type"],
                    "additionalProperties": False,
                },
            },
            "relationships": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "source": {"type": "string"},
                        "relation": {"type": "string", "enum": list(relation_types)},
                        "target": {"type": "string"},
                    },
                    "required": ["source", "relation", "target"],
                    "additionalProperties": False,
                },
            },
        },
        "required": ["entities", "relationships"],
        "additionalProperties": False,
    }

SCHEMA = build_schema(ENTITY_TYPES, RELATION_TYPES)

def build_prompt(text):
    types_txt = "\n".join(f"- {k}: {v}" for k, v in ENTITY_TYPES.items())
    rels_txt = "\n".join(f"- {k}: {v}" for k, v in RELATION_TYPES.items())
    return f"""You are an information extraction system. Build a knowledge graph from the encyclopedia-style TEXT below.

The TEXT begins with the title of its article ("Title: ..."). The article is about the entity named in the title, so "he", "she", "it", "the film" or "the band" refer to that entity.

ENTITY TYPES:
{types_txt}

RELATIONS, written as (source, RELATION, target). Use only these:
{rels_txt}

Rules:
1. Extract only what the TEXT explicitly states. Never use outside knowledge, even if you know more about the subject.
2. Entity names are noun phrases taken from the text or the title, using the fullest name given. Never use verbs or actions.
3. Follow the relation definitions exactly. They say which way each relation points.
4. Dates are DATE entities, copied as written.
5. Every source and target of a relationship must also appear in the entities list.
6. Prefer a specific relation over RELATED_TO. If nothing fits, return empty lists.

TEXT:
{text}
"""

def extract(text, model=None):
    response = chat_with_retry(
        model=model or LLM,
        messages=[
            {
                "role": "user",
                "content": build_prompt(text)
            }
        ],
        temperature=0,
        reasoning_effort="low",
        max_completion_tokens=4096,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "graph_extraction",
                "strict": True,
                "schema": SCHEMA
            }
        }
    )

    content = response.choices[0].message.content

    if not content:
        raise ValueError("The model returned empty content.")

    return json.loads(content)

def validate_extraction(data):
    if not isinstance(data, dict):
        return False
    if not isinstance(data.get("entities"), list) or not isinstance(data.get("relationships"), list):
        return False
    for e in data["entities"]:
        if not isinstance(e, dict) or "name" not in e or "type" not in e:
            return False
    for r in data["relationships"]:
        if not isinstance(r, dict) or not {"source", "relation", "target"} <= r.keys():
            return False
    return True

def normalize_entity_name(name):
    return re.sub(r"\s+", " ", str(name)).strip()

def normalize_extraction(data):
    entities, relationships = [], []
    seen_e, seen_r = set(), set()
    for e in data.get("entities", []):
        name = normalize_entity_name(e["name"])
        etype = str(e["type"]).strip().upper()
        if name and (name.lower(), etype) not in seen_e:
            entities.append({"name": name, "type": etype})
            seen_e.add((name.lower(), etype))
    for r in data.get("relationships", []):
        s = normalize_entity_name(r["source"])
        rel = str(r["relation"]).strip().upper()
        t = normalize_entity_name(r["target"])
        if s and t and rel and (s.lower(), rel, t.lower()) not in seen_r:
            relationships.append({"source": s, "relation": rel, "target": t})
            seen_r.add((s.lower(), rel, t.lower()))
    return {"entities": entities, "relationships": relationships}

In [21]:
TEST_TITLES = ["Naïma Ben Ali", "Zine El Abidine Ben Ali", "Gyrid of Sweden", "Harald Bluetooth"]
by_title = {c["title"]: c for c in chunks}

def show_extraction(ch):
    print(f"\n===== {ch['title']} [{ch['id']}] =====")

    try:
        out = extract_with_retry(ch["text"])

        print("Valid shape:", validate_extraction(out))

        print("ENTITIES:")
        for e in out["entities"]:
            print(f"  {e['name']} [{e['type']}]")

        print("RELATIONSHIPS:")
        for r in out["relationships"]:
            print(
                f"  {r['source']} "
                f"--{r['relation']}--> {r['target']}"
            )

        return out

    except RuntimeError as e:
        print("FAILED:", e)
        return None

In [22]:
show_extraction(by_title[TEST_TITLES[0]])


===== Naïma Ben Ali [0101_na_ma_ben_ali] =====
Valid shape: True
ENTITIES:
  Naïma Ben Ali [PERSON]
  Zine El Abidine Ben Ali [PERSON]
  Tunisia [PLACE]
  First Lady of Tunisia [OTHER]
RELATIONSHIPS:
  Naïma Ben Ali --SPOUSE--> Zine El Abidine Ben Ali
  Naïma Ben Ali --RELATED_TO--> First Lady of Tunisia


{'entities': [{'name': 'Naïma Ben Ali', 'type': 'PERSON'},
  {'name': 'Zine El Abidine Ben Ali', 'type': 'PERSON'},
  {'name': 'Tunisia', 'type': 'PLACE'},
  {'name': 'First Lady of Tunisia', 'type': 'OTHER'}],
 'relationships': [{'source': 'Naïma Ben Ali',
   'relation': 'SPOUSE',
   'target': 'Zine El Abidine Ben Ali'},
  {'source': 'Naïma Ben Ali',
   'relation': 'RELATED_TO',
   'target': 'First Lady of Tunisia'}]}

In [23]:
results = {}

for title in TEST_TITLES:
    print(f"\n{'='*60}")
    result = show_extraction(by_title[title])
    results[title] = result



===== Naïma Ben Ali [0101_na_ma_ben_ali] =====
Valid shape: True
ENTITIES:
  Naïma Ben Ali [PERSON]
  Zine El Abidine Ben Ali [PERSON]
  First Lady of Tunisia [OTHER]
RELATIONSHIPS:
  Naïma Ben Ali --SPOUSE--> Zine El Abidine Ben Ali
  Naïma Ben Ali --RELATED_TO--> First Lady of Tunisia


===== Zine El Abidine Ben Ali [0103_zine_el_abidine_ben_ali] =====
Structured-output failure (attempt 1/3): Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}
Structured-output failure (attempt 2/3): Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}
Structured-output failure (attempt 3/3): Error code: 400 - {'error': {'message': "Failed to validate JS

In [26]:
FAILED_TITLES = [
    "Zine El Abidine Ben Ali",
    "Harald Bluetooth"
]

for titles in FAILED_TITLES:
  ch = by_title[title]

  print("TITLE:", title)
  print("CHUNK ID:", ch["id"])
  print("CHARACTERS:", len(ch["text"]))
  print("\nACTUAL TEXT:\n")
  print(ch["text"])

TITLE: Harald Bluetooth
CHUNK ID: 0141_harald_bluetooth
CHARACTERS: 589

ACTUAL TEXT:

Harald Bluetooth: Harald "Bluetooth" Gormsson (died c. 985/86) was a king of Denmark and Norway. He was the son of King Gorm the Old and of Thyra Dannebod. Harald ruled as king of Denmark from c. 958 – c. 986. Harald introduced Christianity to Denmark and consolidated his rule over most of Jutland and Zealand. Harald's rule as king of Norway following the assassination of King Harald Greycloak of Norway was more tenuous, most likely lasting for no more than a few years in the 970s. Some sources say his son Sweyn Forkbeard forcibly deposed him from his Danish throne before his death.
TITLE: Harald Bluetooth
CHUNK ID: 0141_harald_bluetooth
CHARACTERS: 589

ACTUAL TEXT:

Harald Bluetooth: Harald "Bluetooth" Gormsson (died c. 985/86) was a king of Denmark and Norway. He was the son of King Gorm the Old and of Thyra Dannebod. Harald ruled as king of Denmark from c. 958 – c. 986. Harald introduced Christia

In [27]:
def extract_json_mode(text):
    response = chat_with_retry(
        model=LLM,
        messages=[
            {
                "role": "user",
                "content": build_prompt(text)
                + "\n\nReturn only valid JSON with exactly "
                  "two keys: entities and relationships."
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    raw = response.choices[0].message.content

    print("RAW MODEL RESPONSE:\n")
    print(raw)

    data = json.loads(raw)

    print("\nSTRUCTURAL VALIDATION:",
          validate_extraction(data))

    return normalize_extraction(data)

In [37]:
harald = by_title["Harald Bluetooth"]

test_response = groq_client.chat.completions.create(
    model=LLM,
    messages=[
        {
            "role": "user",
            "content": build_prompt(harald["text"])
        }
    ],
    temperature=0,
    max_completion_tokens=1500
)

message = test_response.choices[0].message

print("FINISH REASON:", test_response.choices[0].finish_reason)
print("CONTENT:", repr(message.content))
print("REASONING:", repr(message.reasoning))
print("USAGE:", test_response.usage)

FINISH REASON: length
CONTENT: ''
REASONING: 'We need to build knowledge graph. Entities: Harald Bluetooth, King Gorm the Old, Thyra Dannebod, Denmark, Norway, Jutland, Zealand, King Harald Greycloak of Norway, Sweyn Forkbeard. Also maybe "Christianity" but it\'s a concept, not a person or work. But we can include as OTHER? But relation? "introduced Christianity to Denmark" could be a relation: source Harald Bluetooth, relation ??? There\'s no relation defined for "introduced". We could use RELATED_TO. But rule: prefer specific relation over RELATED_TO. No specific relation for "introduced". So use RELATED_TO: (Harald Bluetooth, RELATED_TO, Christianity). But Christianity is not defined as entity type. We can create OTHER entity "Christianity". Also "c. 985/86" is a date. "c. 958 – c. 986" is a date range; we can treat as two dates? The rule: DATE entities copied as written. So we can create DATE "c. 985/86", "c. 958", "c. 986". Also "970s" is a date? It\'s a decade. We can treat as DA

In [38]:
def extract_without_json_mode(text):
    response = chat_with_retry(
        model=LLM,
        messages=[
            {
                "role": "user",
                "content": build_prompt(text)
                + "\n\nReturn only valid JSON. Do not include "
                  "Markdown or explanations."
            }
        ],
        temperature=0
    )

    raw = response.choices[0].message.content

    print("RAW MODEL RESPONSE:\n")
    print(raw)

    return raw

In [32]:
harald = by_title["Harald Bluetooth"]

harald_raw = extract_without_json_mode(harald["text"])

RAW MODEL RESPONSE:




In [39]:
test_response = groq_client.chat.completions.create(
    model=LLM,
    messages=[
        {
            "role": "user",
            "content": build_prompt(harald["text"])
        }
    ],
    temperature=0,
    max_completion_tokens=1500
)

message = test_response.choices[0].message

print("FINISH REASON:", test_response.choices[0].finish_reason)
print("CONTENT:", repr(message.content))
print("REASONING:", repr(message.reasoning))
print("USAGE:", test_response.usage)

FINISH REASON: length
CONTENT: ''
REASONING: 'We need to build knowledge graph. Entities: Harald Bluetooth, King Gorm the Old, Thyra Dannebod, Denmark, Norway, Jutland, Zealand, King Harald Greycloak of Norway, Sweyn Forkbeard. Also maybe "Christianity" but it\'s a concept, not a person or work. But we can include as OTHER? But relation? "introduced Christianity to Denmark" could be a relation: source Harald Bluetooth, relation ??? There\'s no relation defined for "introduced". So we might use RELATED_TO. But rule: prefer specific relation over RELATED_TO. No specific relation for "introduced". So use RELATED_TO: (Harald Bluetooth, RELATED_TO, Christianity). But Christianity is not defined as entity type; we can use OTHER. But we need to include entity "Christianity" as OTHER. Also "c. 985/86" is a date. "c. 958 – c. 986" is a date range; we can treat as two dates? The rule: DATE entities copied as written. So we can have DATE "c. 985/86", DATE "c. 958", DATE "c. 986". Also "970s" is a

In [41]:
test_response = groq_client.chat.completions.create(
    model=LLM,
    messages=[
        {
            "role": "user",
            "content": build_prompt(harald["text"])
        }
    ],
    temperature=0,
    reasoning_effort="low",
    max_completion_tokens=4096,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "graph_extraction",
            "strict": True,
            "schema": SCHEMA
        }
    }
)

message = test_response.choices[0].message

print("FINISH REASON:", test_response.choices[0].finish_reason)
print("CONTENT:", repr(message.content))

if message.content:
    data = json.loads(message.content)
    print("\nSTRUCTURAL VALIDATION:", validate_extraction(data))
    print("\nEXTRACTED RESULT:")
    print(json.dumps(data, indent=2))

print("\nUSAGE:", test_response.usage)

FINISH REASON: stop
CONTENT: '{"entities":[{"name":"Harald Bluetooth","type":"PERSON"},{"name":"King Gorm the Old","type":"PERSON"},{"name":"Thyra Dannebod","type":"PERSON"},{"name":"Sweyn Forkbeard","type":"PERSON"},{"name":"King Harald Greycloak","type":"PERSON"},{"name":"Denmark","type":"PLACE"},{"name":"Norway","type":"PLACE"},{"name":"Jutland","type":"PLACE"},{"name":"Zealand","type":"PLACE"},{"name":"Christianity","type":"OTHER"},{"name":"c. 985/86","type":"DATE"},{"name":"c. 958","type":"DATE"},{"name":"c. 986","type":"DATE"},{"name":"970s","type":"DATE"}],"relationships":[{"source":"Harald Bluetooth","relation":"FATHER","target":"King Gorm the Old"},{"source":"Harald Bluetooth","relation":"MOTHER","target":"Thyra Dannebod"},{"source":"Harald Bluetooth","relation":"DATE_OF_DEATH","target":"c. 985/86"},{"source":"Harald Bluetooth","relation":"CHILD","target":"Sweyn Forkbeard"},{"source":"Harald Bluetooth","relation":"RELATED_TO","target":"Denmark"},{"source":"Harald Bluetooth","r

In [43]:
harald_result = extract(harald["text"])

print(json.dumps(harald_result, indent=2))

{
  "entities": [
    {
      "name": "Harald Bluetooth",
      "type": "PERSON"
    },
    {
      "name": "King Gorm the Old",
      "type": "PERSON"
    },
    {
      "name": "Thyra Dannebod",
      "type": "PERSON"
    },
    {
      "name": "Denmark",
      "type": "PLACE"
    },
    {
      "name": "Norway",
      "type": "PLACE"
    },
    {
      "name": "Jutland",
      "type": "PLACE"
    },
    {
      "name": "Zealand",
      "type": "PLACE"
    },
    {
      "name": "King Harald Greycloak",
      "type": "PERSON"
    },
    {
      "name": "Sweyn Forkbeard",
      "type": "PERSON"
    },
    {
      "name": "c. 985/86",
      "type": "DATE"
    },
    {
      "name": "c. 958",
      "type": "DATE"
    },
    {
      "name": "c. 986",
      "type": "DATE"
    },
    {
      "name": "970s",
      "type": "DATE"
    }
  ],
  "relationships": [
    {
      "source": "Harald Bluetooth",
      "relation": "FATHER",
      "target": "King Gorm the Old"
    },
    {
      "source

In [44]:
print(harald["text"])

Harald Bluetooth: Harald "Bluetooth" Gormsson (died c. 985/86) was a king of Denmark and Norway. He was the son of King Gorm the Old and of Thyra Dannebod. Harald ruled as king of Denmark from c. 958 – c. 986. Harald introduced Christianity to Denmark and consolidated his rule over most of Jutland and Zealand. Harald's rule as king of Norway following the assassination of King Harald Greycloak of Norway was more tenuous, most likely lasting for no more than a few years in the 970s. Some sources say his son Sweyn Forkbeard forcibly deposed him from his Danish throne before his death.
